# 03 — Churn Prediction: Logistic Regression, Decision Tree & Random Forest

In [2]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
DATA=ROOT/'C:/Users/hp/OneDrive/Documents/New folder (2)/Telco-Customer-Churn.csv'
df=pd.read_csv(DATA)
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df=df.dropna(subset=['TotalCharges']).copy()
df['ChurnFlag']=(df['Churn']=='Yes').astype(int)
df['ServicesCount']=(df[['PhoneService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']].eq('Yes')).sum(axis=1)


## 1. Train/test split and preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
features=[c for c in df.columns if c not in ['customerID','Churn','ChurnFlag','TenureBand']]
X=df[features]; y=df.ChurnFlag
cat=X.select_dtypes(include='object').columns.tolist(); num=X.select_dtypes(exclude='object').columns.tolist()
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)

## 2. Models and evaluation

In [4]:
models={'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced'),'Decision Tree':DecisionTreeClassifier(max_depth=5,min_samples_leaf=20,class_weight='balanced',random_state=42),'Random Forest':RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1)}
results=[]
for name,m in models.items():
    pipe=Pipeline([('pre',pre),('model',m)]); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); prob=pipe.predict_proba(Xte)[:,1]
    results.append([name,accuracy_score(yte,pred),precision_score(yte,pred),recall_score(yte,pred),f1_score(yte,pred),roc_auc_score(yte,prob)])
    print(name); print(confusion_matrix(yte,pred)); print(classification_report(yte,pred))
res=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1','ROC_AUC']); display(res)

Logistic Regression
[[723 310]
 [ 76 298]]
              precision    recall  f1-score   support

           0       0.90      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407

Decision Tree
[[739 294]
 [ 77 297]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1033
           1       0.50      0.79      0.62       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.74      0.75      1407

Random Forest
[[774 259]
 [ 91 283]]
              precision    recall  f1-score   support

           0       0.89      0.75      0.82      1033
           1       0.52      0.76      0.62       374

    accuracy                           0.75      1407
   macro av

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.725657,0.490132,0.796791,0.606925,0.835116
1,Decision Tree,0.736318,0.502538,0.794118,0.615544,0.827742
2,Random Forest,0.751244,0.522140,0.756684,0.617904,0.832738


## 3. Feature interpretation

In [5]:
# Random-forest permutation importance on the held-out set
from sklearn.inspection import permutation_importance
rf=Pipeline([('pre',pre),('model',models['Random Forest'])]); rf.fit(Xtr,ytr)
pi=permutation_importance(rf,Xte,yte,n_repeats=10,random_state=42,scoring='roc_auc')
imp=pd.DataFrame({'feature':Xte.columns,'importance':pi.importances_mean}).sort_values('importance',ascending=False); display(imp.head(15))

,feature,importance
14,Contract,0.056988
4,tenure,0.024038
7,InternetService,0.022088
18,TotalCharges,0.011503
8,OnlineSecurity,0.005680
11,TechSupport,0.005179
15,PaperlessBilling,0.002519
16,PaymentMethod,0.002224
13,StreamingMovies,0.001296
17,MonthlyCharges,0.001142


## Business meaning
Recall is especially important when missing a likely churner is costly. ROC-AUC summarizes ranking ability, while precision/recall at an operational threshold should drive campaign design. Model outputs are predictive associations, not causal explanations.